In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display

# 1) CSV → DataFrame (6행 라벨/7행부터 데이터, '날씨' 제외)
def load_trend_csv(path: str, drop_series=('날씨',)):
    raw = pd.read_csv(path, header=None, encoding='utf-8-sig')
    label_row = raw.iloc[6, :]
    data = raw.iloc[7:, :].reset_index(drop=True)
    date = pd.to_datetime(data.iloc[:, 0], errors='coerce')

    series_specs = []
    for col_idx in range(1, raw.shape[1], 2):
        name = str(label_row[col_idx]).strip()
        if name and name.lower() != 'nan' and name not in drop_series:
            series_specs.append((name, col_idx))

    df = pd.DataFrame({'날짜': date})
    for name, val_idx in series_specs:
        df[name] = pd.to_numeric(data.iloc[:, val_idx], errors='coerce')
    return df.dropna()

# 2) 데이터 로드 & 상관계수
csv_path = "cleaned_no_weather.csv"
df = load_trend_csv(csv_path, drop_series=('날씨',))
corr = df.drop(columns=['날짜']).corr(method='pearson').round(2)

# 3) 파랑·회색 순차형 컬러스케일 (0~1 범위)
sequential_blue_gray = [
    [0.00, "#F5F5F5"],
    [0.25, "#E6E9EF"],
    [0.50, "#C8D0DC"],
    [0.75, "#7A8CA5"],
    [1.00, "#5A6D8C"],
]

# 4) Plotly 히트맵 (칸 내부 텍스트: 크기↑ + 볼드)
fig = go.Figure(
    data=go.Heatmap(
        z=corr.values,
        x=corr.columns.tolist(),
        y=corr.index.tolist(),
        zmin=0, zmax=1,
        colorscale=sequential_blue_gray,
        colorbar=dict(title='상관계수'),
        text=corr.values,
        texttemplate="%{text:.2f}",
        textfont=dict(
            family="Pretendard Variables, system-ui, -apple-system, Segoe UI, Arial",
            size=14,                # 글씨 크기 확대
            color="#222222"         # 텍스트 컬러도 #222222
        )
    )
)

# 외부(제목/축/범례) 폰트 설정
fig.update_layout(
    title="키워드 상관관계 히트맵 (0–1, 파랑·회색 스케일)",
    width=860, height=680,
    font=dict(
        family="Pretendard Variables, system-ui, -apple-system, Segoe UI, Arial",
        size=15,
        color="#222222"           # 외부 폰트 컬러
    ),
    margin=dict(l=0, r=0, t=60, b=0),
)

# 5) Jupyter에서 둥근 모서리 표시
html_inner = pio.to_html(fig, include_plotlyjs='cdn', full_html=False)
html_doc = f"""
<link rel="preconnect" href="https://cdn.jsdelivr.net" crossorigin>
<style>
@font-face {{
  font-family: 'Pretendard Variables';
  src: url('https://cdn.jsdelivr.net/gh/orioncactus/pretendard/dist/web/static/pretendard-std-variable.woff2') format('woff2');
  font-weight: 45 920; font-style: normal; font-display: swap;
}}
.round-box {{
  border-radius: 20px;
  overflow: hidden;
  box-shadow: 0 8px 20px rgba(0,0,0,0.08);
  font-family: 'Pretendard Variables', system-ui, -apple-system, Segoe UI, Arial, sans-serif;
  font-weight: 600;   /* 칸 내부 텍스트 Bold */
}}
</style>
<div class="round-box">{html_inner}</div>
"""
display(HTML(html_doc))
